In [130]:
#imports
from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, BooleanType, BinaryType
from datetime import datetime
import ConnectionConfig as cc
debugging_mode=True


In [131]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_USER",4)
spark.getActiveSession()

run_timestamp = datetime.now()

Environment variables are set...


In [132]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [133]:
#get info
# user tabel
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "user_table") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("users")

# treusure log tabel
df_treasure_log = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_log.createOrReplaceTempView("treasure_log")

#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

In [134]:
#alles in 1 grote ding zetten
df_dim_user_new = spark.sql("""
                            SELECT u.id as userId,
                                   u.first_name as first_Name,
                                   u.last_name as last_Name,
                                   u.mail as email,
                                   u.street || "" || u.number as address,
                                   CASE
                                       WHEN treasure_count is null THEN 'Starter'
                                       WHEN treasure_count < 4 THEN 'Amateur'
                                       WHEN treasure_count BETWEEN 4 AND 10 THEN 'Professional'
                                       ELSE 'Pirate'
                                       END as experienceLevel,
                                   CASE
                                       WHEN EXISTS (SELECT 1 FROM treasure t WHERE t.owner_id = u.id) THEN true
                                       ELSE false
                                       END     as dedicator,
                                   to_timestamp('1990-01-01') as scd_start,
                                   to_timestamp('2100-12-12') as scd_end,
                                   true as current,
                                   md5(concat(  CASE
                                                WHEN tl.treasure_count IS NULL THEN 'Starter'
                                                WHEN tl.treasure_count < 4 THEN 'Amateur'
                                                WHEN tl.treasure_count BETWEEN 4 AND 10 THEN 'Professional'
                                                ELSE 'Pirate'
                                                END)
                                   ) as md5
                            FROM users u
                            LEFT JOIN (
                                SELECT hunter_id, COUNT(*) as treasure_count
                                FROM treasure_log
                                GROUP BY hunter_id
                            ) tl ON u.id = tl.hunter_id
                            """)

In [135]:
#deltatabel maken
spark.sql("DROP TABLE IF EXISTS dimUser")

DeltaTable.createOrReplace(spark) \
    .tableName("dimUser") \
    .addColumn("userSurKey", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("userId", BinaryType(), nullable=False) \
    .addColumn("first_Name", StringType()) \
    .addColumn("last_Name", StringType()) \
    .addColumn("email", StringType()) \
    .addColumn("address", StringType()) \
    .addColumn("experienceLevel", StringType()) \
    .addColumn("dedicator", BooleanType()) \
    .addColumn("scd_start", TimestampType()) \
    .addColumn("scd_end", TimestampType()) \
    .addColumn("md5", StringType()) \
    .addColumn("current", BooleanType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

AnalysisException: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`default`.`dimUser`'). The associated location ('file:/home/jovyan/work/Project/spark-warehouse/dimuser') is not empty and also not a Delta table.

In [136]:
#maken van tabel
df_dim_user_new.write.format("delta").mode("overwrite").saveAsTable("dimUser")
spark.sql("SELECT * FROM dimUser").show(10)

+----------+--------------------+----------+----------+--------------------+--------------------+---------------+---------+-------------------+-------------------+--------------------+-------+
|userSurKey|              userId|first_Name| last_Name|               email|             address|experienceLevel|dedicator|          scd_start|            scd_end|                 md5|current|
+----------+--------------------+----------+----------+--------------------+--------------------+---------------+---------+-------------------+-------------------+--------------------+-------+
|    975499|[00 00 8B 48 6B 8...|  Federico|Mascareñas|Federico.Mascareñ...|    Puente Eloisa785|        Amateur|    false|1990-01-01 00:00:00|2100-12-12 00:00:00|78b07a20e6d1bb02d...|   true|
|    975503|[00 01 2C 54 F0 6...|   Valerie|   Spencer|Valerie.Spencer@d...|       Dare Cliff676|         Pirate|     true|1990-01-01 00:00:00|2100-12-12 00:00:00|95aa99a5810a35a36...|   true|
|    975507|[00 01 3F 56 15 2...|  

Incremetenteren van tabel

In [137]:
#bestaande dimensie inlezen als deze bestaat
dt_dimuser = DeltaTable.forPath(spark,"./spark-warehouse/dimuser")

dt_dimuser.toDF().createOrReplaceTempView("dimUser_current")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
spark.sql("select * from dimUser_current ").show()

+----------+--------------------+----------+----------+--------------------+--------------------+---------------+---------+-------------------+-------------------+--------------------+-------+
|userSurKey|              userId|first_Name| last_Name|               email|             address|experienceLevel|dedicator|          scd_start|            scd_end|                 md5|current|
+----------+--------------------+----------+----------+--------------------+--------------------+---------------+---------+-------------------+-------------------+--------------------+-------+
|    975499|[00 00 8B 48 6B 8...|  Federico|Mascareñas|Federico.Mascareñ...|    Puente Eloisa785|        Amateur|    false|1990-01-01 00:00:00|2100-12-12 00:00:00|78b07a20e6d1bb02d...|   true|
|    975503|[00 01 2C 54 F0 6...|   Valerie|   Spencer|Valerie.Spencer@d...|       Dare Cliff676|         Pirate|     true|1990-01-01 00:00:00|2100-12-12 00:00:00|95aa99a5810a35a36...|   true|
|    975507|[00 01 3F 56 15 2...|  

In [138]:
#maak nieuwe dim user
df_dim_user_new = spark.sql("""
    SELECT
        u.id as source_userId,
        u.first_name as source_first_Name,
        u.last_name as source_last_Name,
        u.mail as source_email,
        CONCAT(u.street, ' ', u.number) as source_address,
        CASE
            WHEN tl.treasure_count IS NULL THEN 'Starter'
            WHEN tl.treasure_count < 4 THEN 'Amateur'
            WHEN tl.treasure_count BETWEEN 4 AND 10 THEN 'Professional'
            ELSE 'Pirate'
        END as source_experienceLevel,
        CASE
            WHEN EXISTS (SELECT 1 FROM treasure t WHERE t.owner_id = u.id) THEN true
            ELSE false
        END as source_dedicator,
        md5(CONCAT(
            CASE
                WHEN tl.treasure_count IS NULL THEN 'Starter'
                WHEN tl.treasure_count < 4 THEN 'Amateur'
                WHEN tl.treasure_count BETWEEN 4 AND 10 THEN 'Professional'
                ELSE 'Pirate'
            END
        )) as source_md5
    FROM users u
    LEFT JOIN (
        SELECT hunter_id, COUNT(*) as treasure_count
        FROM treasure_log
        GROUP BY hunter_id
    ) tl ON u.id = tl.hunter_id
""")

df_dim_user_new.createOrReplaceTempView("dimUser_new")

In [139]:
#wijzegingen detecteren
detectedChanges = spark.sql(f"""
    SELECT
        source.*,
        dwh.userId as dwh_userId,
        dwh.md5 as dwh_md5,
        dwh.current as dwh_current
    FROM dimUser_new source
    LEFT OUTER JOIN dimUser_current dwh
        ON dwh.userId = source.source_userId
        AND dwh.current = true
    WHERE dwh.userId IS NULL
        OR dwh.md5 <> source.source_md5
""")

detectedChanges.createOrReplaceTempView("detectedChanges")

In [140]:
#upsert voorberijden
df_upserts = spark.sql(f"""
    SELECT
        source_userId as userId,
        source_first_Name as first_Name,
        source_last_Name as last_Name,
        source_email as email,
        source_address as address,
        source_experienceLevel as experienceLevel,
        source_dedicator as dedicator,
        to_timestamp('{run_timestamp}') as scd_start,
        to_timestamp('2100-12-12') as scd_end,
        source_md5 as md5,
        true as current
    FROM detectedChanges

    UNION ALL

    SELECT
        dwh_userId as userId,
        dwh.first_Name,
        dwh.last_Name,
        dwh.email,
        dwh.address,
        dwh.experienceLevel,
        dwh.dedicator,
        dwh.scd_start,
        to_timestamp('{run_timestamp}') as scd_end,
        dwh.md5,
        false as current
    FROM detectedChanges dc
    INNER JOIN dimUser_current dwh
        ON dwh.userId = dc.dwh_userId
        AND dwh.current = true
    WHERE dc.dwh_userId IS NOT NULL
""")

df_upserts.createOrReplaceTempView("upserts")

In [141]:
#alles mergen
spark.sql("""
    MERGE INTO dimUser_current AS target
    USING upserts AS source
    ON target.userId = source.userId AND source.current = false AND target.current = true
    WHEN MATCHED THEN
        UPDATE SET scd_end = source.scd_end, current = source.current
    WHEN NOT MATCHED THEN
        INSERT (userId, first_Name, last_Name, email, address, experienceLevel, dedicator, scd_start, scd_end, md5, current)
        VALUES (source.userId, source.first_Name, source.last_Name, source.email, source.address, source.experienceLevel, source.dedicator, source.scd_start, source.scd_end, source.md5, source.current)
""")

### Stap 7: Resultaat controleren
dt_dimuser.toDF().sort("userId", "scd_start").show(100)

+----------+--------------------+----------+-----------+--------------------+--------------------+---------------+---------+--------------------+--------------------+--------------------+-------+
|userSurKey|              userId|first_Name|  last_Name|               email|             address|experienceLevel|dedicator|           scd_start|             scd_end|                 md5|current|
+----------+--------------------+----------+-----------+--------------------+--------------------+---------------+---------+--------------------+--------------------+--------------------+-------+
|    975501|[00 00 1C BF A8 E...|     Yanis|      Andre|Yanis.Andre@girar...|Allée, Voie du Ba...|        Amateur|    false| 1990-01-01 00:00:00|2025-10-05 22:38:...|78b07a20e6d1bb02d...|  false|
|   1463250|[00 00 1C BF A8 E...|     Yanis|      Andre|Yanis.Andre@girar...|Allée, Voie du Ba...|        Amateur|    false|2025-10-05 22:38:...| 2100-12-12 00:00:00|9817dd3b15456672f...|   true|
|    975505|[00 00 2

In [142]:
#end de spark sessie
spark.stop()